In [ ]:
# Topic 4 audit: data quality evidence for S01 (AIID incidents) and S02 (MIT AI Risk Repository)
import re, json
from pathlib import Path
import pandas as pd

# ---- settings ----
AIID_SNAPSHOT_DATE = "2026-11-02"
MIT_VERSION_DATE = "2025-12-03"
SEED, SAMPLE_N = 42, 15

# ---- locate the prepared tables ----
candidates = [Path("/content/week3_project/data/processed"), Path("week3_project/data/processed"), Path("/content")]
PROC = next((p for p in candidates if (p / "prepared_incidents.csv").is_file() and (p / "prepared_risks.csv").is_file()), None)
assert PROC, ("prepared_incidents.csv / prepared_risks.csv not found. Upload both files "
              "(GitHub: data/processed) into /content and run again.")
s01 = pd.read_csv(PROC / "prepared_incidents.csv", dtype="string")
s02 = pd.read_csv(PROC / "prepared_risks.csv", dtype="string")
OUT = PROC.parent / "audit"; OUT.mkdir(exist_ok=True)
M = {}
snap = pd.Timestamp(AIID_SNAPSHOT_DATE)
print("Tables loaded from", PROC)
print("S01 prepared incidents:", s01.shape, "| S02 prepared risks:", s02.shape)

def norm(s): return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(s).lower())).strip()
DOMAIN_STEMS = {1: "discrimination", 2: "privacy", 3: "misinformation", 4: "malicious",
                5: "human", 6: "socioeconomic", 7: "safety"}
def domain_no(v):
    if pd.isna(v): return None
    n = norm(v)
    for k, stem in DOMAIN_STEMS.items():
        if stem in n: return k
    return None

# =========================== 1. COMPLETENESS ===========================
print("\n=== 1. COMPLETENESS (missing values in required fields) ===")
def miss(df, cols):
    return {c: {"missing": int(df[c].isna().sum()), "pct": round(100 * df[c].isna().mean(), 1)} for c in cols if c in df.columns}
M["completeness"] = {"S01": miss(s01, ["record_id", "title", "description", "date", "source_url"]),
                     "S02": miss(s02, ["record_id", "risk_category", "risk_subcategory", "description", "domain", "subdomain"])}
for k in ("S01", "S02"):
    print(f"-- {k} --"); print(pd.DataFrame(M["completeness"][k]).T.to_string())
no_desc = s02[s02["description"].isna()]
print("\nExamples of S02 rows with no description (first 8):")
print(no_desc[[c for c in ["record_id", "risk_category", "risk_subcategory", "domain"] if c in s02.columns]].head(8).to_string(index=False, max_colwidth=40))
if "df_s02" in globals() and "map_s02" in globals():       # only when run in the Week 3 notebook runtime
    lvl = [c for c in df_s02.columns if "level" in str(c).lower()]
    if lvl:
        print(f"\nRaw workbook column '{lvl[0]}' for rows with no description:")
        print(df_s02.loc[df_s02[map_s02['description']].isna(), lvl[0]].value_counts(dropna=False).head(8).to_string())

# =========================== 2. CONSISTENCY ===========================
print("\n=== 2. CONSISTENCY (do related fields agree / follow one format?) ===")
d = pd.to_datetime(s01["date"], errors="coerce")
bad_date = int((d.isna() & s01["date"].notna()).sum())
future = int((d > snap).sum())
id_numeric = int(s01["record_id"].str.fullmatch(r"\d+").fillna(False).sum())
url_ok = int((s01["source_url"] == "https://incidentdatabase.ai/cite/" + s01["record_id"]).sum()) if "source_url" in s01 else None
M["consistency_S01"] = {"unparseable_dates": bad_date, "dates_after_snapshot": future,
                        "numeric_ids": id_numeric, "of_rows": len(s01), "source_url_matches_id": url_ok}
print("S01:", M["consistency_S01"])
dn = s02["domain"].map(domain_no)
unmatched = sorted(s02.loc[s02["domain"].notna() & dn.isna(), "domain"].unique())[:10]
sub_no = s02["subdomain"].str.extract(r"^\s*(\d+)\.\d+")[0].astype("Int64") if "subdomain" in s02 else pd.Series(dtype="Int64")
both = dn.notna() & sub_no.notna()
mismatch = int((dn[both].astype(int) != sub_no[both].astype(int)).sum())
id_fmt = int(s02["record_id"].str.fullmatch(r"\d{2}\.\d{2}\.\d{2}").fillna(False).sum())
M["consistency_S02"] = {"domain_values_not_matching_the_7_domains": unmatched, "domain_vs_subdomain_number_mismatch": mismatch,
                        "rows_checked_for_mismatch": int(both.sum()), "ids_in_dd.dd.dd_format": id_fmt, "of_rows": len(s02)}
print("S02:", M["consistency_S02"])

# =========================== 3. ACCURACY ===========================
print("\n=== 3. ACCURACY (does the prepared data faithfully represent its source?) ===")
rng = pd.Series(range(len(s01))).sample(min(SAMPLE_N, len(s01)), random_state=SEED).tolist()
samp1 = s01.iloc[rng]
if "df_s01" in globals() and "map_s01" in globals():
    raw = df_s01.copy(); raw["_id"] = raw[map_s01["record_id"]].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    raw = raw.drop_duplicates("_id").set_index("_id")
    same_t = same_d = n_chk = 0
    for _, r in samp1.iterrows():
        if r["record_id"] in raw.index:
            n_chk += 1
            same_t += str(raw.loc[r["record_id"], map_s01["title"]]).strip() == str(r["title"]).strip()
            same_d += str(raw.loc[r["record_id"], map_s01["description"]]).strip() == str(r["description"]).strip()
    M["accuracy_pipeline_fidelity"] = {"sample": n_chk, "title_identical_to_source": same_t, "description_identical_to_source": same_d}
    print("Prepared vs raw source for the sampled records:", M["accuracy_pipeline_fidelity"])
else:
    print("Raw file not loaded in this runtime, so prepared-vs-source comparison was skipped.")
print("The source's own correctness cannot be measured automatically. MANUAL CHECK: open these 3 links and compare the title and date:")
for _, r in samp1.head(3).iterrows():
    print(" ", r.get("source_url"), "|", str(r["title"])[:70], "|", r["date"])

# =========================== 4. TIMELINESS ===========================
print("\n=== 4. TIMELINESS ===")
yr = d.dt.year.value_counts().sort_index()
latest = d.max()
M["timeliness"] = {"S01_earliest": str(d.min().date()), "S01_latest": str(latest.date()), "S01_snapshot_date": AIID_SNAPSHOT_DATE,
                   "S01_months_latest_incident_to_snapshot": round((snap - latest).days / 30.4, 1),
                   "S02_version_date": MIT_VERSION_DATE,
                   "S02_months_old_at_snapshot": round((snap - pd.Timestamp(MIT_VERSION_DATE)).days / 30.4, 1)}
print(M["timeliness"]); print("Incidents per year (last 8):", yr.tail(8).to_dict())

# =========================== 5. UNIQUENESS ===========================
print("\n=== 5. UNIQUENESS ===")
t_norm = s01["title"].map(norm)
M["uniqueness"] = {"S01_exact_duplicate_rows": int(s01.duplicated().sum()), "S01_duplicate_ids": int(s01["record_id"].duplicated().sum()),
                   "S01_duplicate_normalised_titles": int(t_norm[s01["title"].notna()].duplicated().sum()),
                   "S02_exact_duplicate_rows": int(s02.duplicated().sum()), "S02_duplicate_ids": int(s02["record_id"].duplicated().sum()),
                   "S02_duplicate_descriptions": int(s02["description"].dropna().duplicated().sum())}
print(M["uniqueness"])
dup_ids = s02.loc[s02["record_id"].duplicated(keep=False), ["record_id", "risk_category", "description", "domain"]].sort_values("record_id")
if len(dup_ids): print("\nS02 rows sharing an ID (compare them to see if they are true duplicates):"); print(dup_ids.head(12).to_string(index=False, max_colwidth=45))

# =========================== VALIDATION RULES ===========================
print("\n=== VALIDATION RULE 1: every record has a non-empty, unique record_id (both tables) ===")
r1 = {}
for k, df in (("S01", s01), ("S02", s02)):
    empty = int(df["record_id"].isna().sum()); dup = int(df["record_id"].duplicated().sum())
    r1[k] = {"rows": len(df), "empty_ids": empty, "duplicate_ids": dup, "result": "PASS" if empty == 0 and dup == 0 else "FAIL"}
    print(k, r1[k])
    if dup: print("   duplicate ids (first 10):", df.loc[df["record_id"].duplicated(keep=False), "record_id"].unique()[:10].tolist())
M["rule1"] = r1
print("\n=== VALIDATION RULE 2: every risk record has a non-empty description AND a recognised domain (S02) ===")
no_d, no_dom = s02["description"].isna(), dn.isna()
r2 = {"rows": len(s02), "missing_description": int(no_d.sum()), "missing_or_unrecognised_domain": int(no_dom.sum()),
      "fail_either": int((no_d | no_dom).sum()), "fail_both": int((no_d & no_dom).sum()),
      "pass_rows": int((~no_d & ~no_dom).sum())}
r2["result"] = "PASS" if r2["fail_either"] == 0 else "FAIL"
M["rule2"] = r2; print(r2)

# =========================== COVERAGE AND BIAS ===========================
print("\n=== COVERAGE (S02 risk domains) ===")
dom_counts = dn.dropna().astype(int).map(DOMAIN_STEMS).value_counts()
print(dom_counts.to_string()); M["coverage_S02_domains"] = {k: int(v) for k, v in dom_counts.items()}
print("\n=== COVERAGE (S01 incidents over time) ===")
recent3 = int((d.dt.year >= d.dt.year.max() - 2).sum())
print("Share of incidents from the latest 3 years:", round(100 * recent3 / d.notna().sum(), 1), "%")
M["coverage_S01_recent3yr_pct"] = round(100 * recent3 / d.notna().sum(), 1)
print("\n=== BIAS PROXY (S01): region keywords found in title + description (rough proxy, not a true region field) ===")
REG = {"United States": r"\bunited states\b|\bu\.s\.|\bamerican?\b|\bus\b", "United Kingdom": r"\buk\b|\bunited kingdom\b|\bbritish\b|\bengland\b",
       "China": r"\bchina\b|\bchinese\b", "India": r"\bindia\b|\bindian\b", "Europe/EU": r"\beurope\b|\beuropean\b|\beu\b",
       "Malaysia": r"\bmalaysia\b|\bmalaysian\b", "Africa": r"\bafrica\b|\bafrican\b", "Latin America": r"\bbrazil\b|\bmexico\b|\blatin america\b"}
txt = (s01["title"].fillna("") + " " + s01["description"].fillna("")).str.lower()
reg = {k: int(txt.str.contains(p, regex=True).sum()) for k, p in REG.items()}
print(reg, "| rows scanned:", len(txt)); M["bias_region_keyword_counts"] = reg

# =========================== RESPONSIBLE DATA ===========================
print("\n=== PERSONAL / SENSITIVE CONTENT SCAN (S01 text) ===")
emails = int(txt.str.contains(r"[\w.+-]+@[\w-]+\.[\w.]+", regex=True).sum())
phones = int(txt.str.contains(r"\+?\d[\d\s().-]{8,}\d", regex=True).sum())
sens = int(txt.str.contains(r"\bchild|\bminor|suicide|sexual|abuse|self-harm", regex=True).sum())
M["privacy_scan"] = {"rows_with_email_pattern": emails, "rows_with_phone_pattern": phones, "rows_with_sensitive_topic_words": sens, "rows": len(txt)}
print(M["privacy_scan"])

# =========================== SAMPLES FOR THE APPENDIX ===========================
ed1 = [s01["description"].str.len().idxmax(), s01["description"].str.len().idxmin(), int(d.idxmin()), int(d.idxmax())]
samp_s01 = pd.concat([samp1, s01.loc[ed1]]).drop_duplicates("record_id")
samp_s01 = samp_s01.assign(description=samp_s01["description"].str.slice(0, 110))[["record_id", "title", "date", "description", "source_url"]]
ed2 = list(no_desc.index[:2]) + list(dup_ids.index[:2]) + [s02["description"].str.len().idxmax()]
samp2 = s02.iloc[pd.Series(range(len(s02))).sample(min(SAMPLE_N, len(s02)), random_state=SEED).tolist()]
samp_s02 = pd.concat([samp2, s02.loc[ed2]]).drop_duplicates()
cols2 = [c for c in ["record_id", "risk_category", "domain", "subdomain", "description"] if c in s02.columns]
samp_s02 = samp_s02.assign(description=samp_s02["description"].str.slice(0, 110))[cols2]
samp_s01.to_csv(OUT / "audit_sample_S01.csv", index=False); samp_s02.to_csv(OUT / "audit_sample_S02.csv", index=False)
(OUT / "audit_metrics.json").write_text(json.dumps(M, indent=2, default=str), encoding="utf-8")
print("\n=== SAMPLE S01 (random + edge cases) ==="); print(samp_s01.drop(columns="description").to_string(index=False, max_colwidth=60))
print("\n=== SAMPLE S02 (random + edge cases) ==="); print(samp_s02.drop(columns="description").to_string(index=False, max_colwidth=40))
print("\nSaved evidence files in", OUT, ":", [p.name for p in OUT.iterdir()])


Tables loaded from /content
S01 prepared incidents: (1786, 8) | S02 prepared risks: (2574, 9)

=== 1. COMPLETENESS (missing values in required fields) ===
-- S01 --
             missing  pct
record_id        0.0  0.0
title            0.0  0.0
description      0.0  0.0
date             0.0  0.0
source_url       0.0  0.0
-- S02 --
                  missing   pct
record_id             0.0   0.0
risk_category        74.0   2.9
risk_subcategory    718.0  27.9
description         926.0  36.0
domain              972.0  37.8
subdomain           736.0  28.6

Examples of S02 rows with no description (first 8):
 record_id                       risk_category risk_subcategory domain
  01.00.00                                <NA>             <NA>   <NA>
01.01.00.a Type 1: Diffusion of responsibility             <NA>   <NA>
01.01.00.b Type 1: Diffusion of responsibility             <NA>   <NA>
01.01.00.c Type 1: Diffusion of responsibility             <NA>   <NA>
01.02.00.a        Type 2: Bigger than

In [ ]:
import re
t = (s01["title"].fillna("") + " " + s01["description"].fillna(""))
hit = t[t.str.contains(r"[\w.+-]+@[\w-]+\.[\w.]+", regex=True)]
for i in hit.index:
    print("record_id:", s01.loc[i, "record_id"])
    for m in re.finditer(r"[\w.+-]+@[\w-]+\.[\w.]+", t[i]):
        a = max(0, m.start() - 80)
        print("...", t[i][a:m.end() + 80], "...")

record_id: 1680
...  On February 17, 2026, a still-valid npm token was used to publish unauthorized cline@2.3.0, which installed OpenClaw without user intent. Cline said OpenClaw was non-mali ...
